# Segmented Embedding Retrieval Pipeline

## Project Goal

The goal of this component is to build and evaluate a **multi-field academic paper retrieval pipeline**.

Instead of representing each paper with a single embedding, we represent it with **separate embeddings for different textual fields**. This lets us preserve distinctions between where information appears in a paper — for example, in the title, abstract, body, or conclusion.

We will compare this structured method against a standard **title + abstract baseline**.

---

## Embedding

Each paper will be represented as a tuple of field-specific embeddings:

- **title embedding**
- **abstract embedding**
- **body embedding**
- **conclusion embedding**

All fields will be embedded using the same pretrained model:

**`nomic-embed-text-v1.5`**

At query time, the query will also be embedded, compared against each field separately, and then combined into a final retrieval score.

---

## Plan

### 1. Load the paper dataset

For each paper, we want access to:

- paper ID
- title
- abstract
- full text, if available

This will give us the raw material needed for segmentation and embedding.

---

### 2. Segment each paper into structured fields

For each paper, we will extract:

- **Title**
- **Abstract**
- **Body**
- **Conclusion**

If a conclusion section is not explicitly available, we will use a fallback strategy such as taking the final section or final chunk of the paper.

For the body, we may chunk the text into smaller pieces and aggregate them afterward.

---

### 3. Generate field-specific embeddings

Using the Nomic embedding model, we will encode each field separately.

For every paper, we will store:

- `title_emb`
- `abstract_emb`
- `body_emb`
- `conclusion_emb`


For each paper, we compute a weighted combination of query-to-field similarities.

Conceptually:

`score(query, paper) = wt * sim(query, title) + wa * sim(query, abstract) + wb * sim(query, body) + wc * sim(query, conclusion)`

We will begin with simple fixed weights, and later optionally test tuned or learned weights.




# Data Prep & Exploration

In [1]:
from pathlib import Path
import json
import pandas as pd
import os

ARXIV_JSON = Path("/scratch/mmd9604/arxiv/arxiv-metadata-oai-snapshot.json")
print(ARXIV_JSON.exists(), ARXIV_JSON)

True /scratch/mmd9604/arxiv/arxiv-metadata-oai-snapshot.json


In [2]:
RAW_FIELDS = [
    "id",
    "submitter",
    "authors",
    "title",
    "comments",
    "journal-ref",
    "doi",
    "report-no",
    "categories",
    "license",
    "abstract",
    "versions",
    "update_date",
    "authors_parsed",
]

def stream_arxiv_json(path, max_records=None):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if max_records is not None and i >= max_records:
                break
            record = json.loads(line)
            yield {k: record.get(k) for k in RAW_FIELDS}

In [3]:
def clean_text(x):
    if not x:
        return ""
    return " ".join(str(x).strip().split())

def normalize_record(rec):
    return {
        "id": rec.get("id"),
        "submitter": clean_text(rec.get("submitter")),
        "authors": clean_text(rec.get("authors")),
        "title": clean_text(rec.get("title")),
        "comments": clean_text(rec.get("comments")),
        "journal_ref": clean_text(rec.get("journal-ref")),
        "doi": clean_text(rec.get("doi")),
        "report_no": clean_text(rec.get("report-no")),
        "categories": clean_text(rec.get("categories")),
        "license": rec.get("license"),
        "abstract": clean_text(rec.get("abstract")),
        "versions": rec.get("versions"),
        "update_date": rec.get("update_date"),
        "authors_parsed": rec.get("authors_parsed"),
    }

In [4]:
import pandas as pd

probe_n = 1000
probe_records = [normalize_record(rec) for rec in stream_arxiv_json(ARXIV_JSON, max_records=probe_n)]
probe_df = pd.DataFrame(probe_records)

probe_df.head(3)

,id,submitter,authors,title,comments,journal_ref,doi,report_no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturbati...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,,,,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-pe...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",,,,physics.gen-ph,None,The evolution of Earth-Moon system is describe...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"


In [5]:
# Missingness (fraction of null / empty values)
missing = probe_df.apply(
    lambda col: (col.isna() | (col == "")).mean()
).sort_values(ascending=False)

print("Missingness by field:")
print(missing)


# Length stats for key text fields
def avg_len(series):
    return series.dropna().apply(lambda x: len(x.split())).mean()

print("\nAverage token length:")
print({
    "title": avg_len(probe_df["title"]),
    "abstract": avg_len(probe_df["abstract"]),
    "comments": avg_len(probe_df["comments"]),
})


# Versions / authors structure
print("\nAverage number of versions:")
print(probe_df["versions"].apply(lambda x: len(x) if isinstance(x, list) else 0).mean())

print("\nAverage number of authors:")
print(probe_df["authors_parsed"].apply(lambda x: len(x) if isinstance(x, list) else 0).mean())

Missingness by field:
license           0.939
report_no         0.922
journal_ref       0.427
doi               0.337
comments          0.099
submitter         0.000
authors           0.000
title             0.000
id                0.000
categories        0.000
abstract          0.000
versions          0.000
update_date       0.000
authors_parsed    0.000
dtype: float64

Average token length:
{'title': np.float64(9.551), 'abstract': np.float64(121.013), 'comments': np.float64(9.214)}

Average number of versions:
1.578

Average number of authors:
3.068


In [6]:
def build_segmented_fields(row):
    return {
        "title": row["title"],
        "abstract": row["abstract"],
        "comments": row["comments"] if row["comments"] else ""
    }

segmented_sample = probe_df.apply(build_segmented_fields, axis=1)

segmented_sample.iloc[0]

{'title': 'Calculation of prompt diphoton production cross sections at Tevatron and LHC energies',
 'abstract': 'A fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders. All next-to-leading order perturbative contributions from quark-antiquark, gluon-(anti)quark, and gluon-gluon subprocesses are included, as well as all-orders resummation of initial-state gluon radiation valid at next-to-next-to-leading logarithmic accuracy. The region of phase space is specified in which the calculation is most reliable. Good agreement is demonstrated with data from the Fermilab Tevatron, and predictions are made for more detailed tests with CDF and DO data. Predictions are shown for distributions of diphoton pairs produced at the energy of the Large Hadron Collider (LHC). Distributions of the diphoton pairs from the decay of a Higgs boson are contrasted with those produced from QCD processes at the LHC, showin

In [7]:
def parse_categories(cat_str):
    if not cat_str:
        return [], None
    cats = cat_str.split()
    return cats, cats[0]

probe_df["all_categories"], probe_df["primary_category"] = zip(
    *probe_df["categories"].apply(parse_categories)
)

probe_df[["categories", "all_categories", "primary_category"]].head(3)

,categories,all_categories,primary_category
0,hep-ph,[hep-ph],hep-ph
1,math.CO cs.CG,"[math.CO, cs.CG]",math.CO
2,physics.gen-ph,[physics.gen-ph],physics.gen-ph


In [17]:
probe_df["baseline_text"] = probe_df["title"] + " " + probe_df["abstract"]
probe_df[["title", "baseline_text"]].head(2)

,title,baseline_text
0,Calculation of prompt diphoton production cros...,Calculation of prompt diphoton production cros...
1,Sparsity-certifying Graph Decompositions,Sparsity-certifying Graph Decompositions We de...


## Multi-Field Retrieval Scoring

Each paper is represented using separate embeddings for different fields:

- title  
- abstract  
- comments  

Given a query, we compute a single query embedding and compare it to each field independently.

### Score

score(q, d) =  
w_title * cos(q, title)  
+ w_abstract * cos(q, abstract)  
+ w_comments * cos(q, comments)

### Notes

- `cos` = cosine similarity  
- weights control the importance of each field  
- fields contribute independently to the final ranking  

We do **not** include categories in the embedding — they are used only as metadata for filtering and evaluation.

In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True
)

<All keys matched successfully>


In [15]:
def embed_texts(texts, prefix):
    # nomic expects task prefixes
    texts = [f"{prefix}: {t}" for t in texts]
    return model.encode(texts, normalize_embeddings=True)

    

In [18]:
# test on a small batch
texts = probe_df["baseline_text"].tolist()[:32]

embs = embed_texts(texts, prefix="search_document")

print(len(embs), embs[0].shape)

32 (768,)


In [19]:
batch = probe_df.iloc[:32]

title_embs = embed_texts(batch["title"].tolist(), prefix="search_document")
abstract_embs = embed_texts(batch["abstract"].tolist(), prefix="search_document")
comments_embs = embed_texts(batch["comments"].fillna("").tolist(), prefix="search_document")

print(title_embs.shape, abstract_embs.shape, comments_embs.shape)

(32, 768) (32, 768) (32, 768)


In [20]:
paper_embs_0 = {
    "title": title_embs[0],
    "abstract": abstract_embs[0],
    "comments": comments_embs[0],
}

{k: v.shape for k, v in paper_embs_0.items()}

{'title': (768,), 'abstract': (768,), 'comments': (768,)}

In [21]:
paper_embs = []

for i in range(len(batch)):
    paper_embs.append({
        "paper_id": batch.iloc[i]["id"],
        "title": title_embs[i],
        "abstract": abstract_embs[i],
        "comments": comments_embs[i],
    })

paper_embs[0].keys()

dict_keys(['paper_id', 'title', 'abstract', 'comments'])

In [22]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b)  # already normalized

weights = {
    "title": 0.4,
    "abstract": 0.5,
    "comments": 0.1
}

query = "graph neural networks for molecules"

query_emb = embed_texts([query], prefix="search_query")[0]

scores = []

for p in paper_embs:
    score = (
        weights["title"] * cosine_similarity(query_emb, p["title"]) +
        weights["abstract"] * cosine_similarity(query_emb, p["abstract"]) +
        weights["comments"] * cosine_similarity(query_emb, p["comments"])
    )
    scores.append((p["paper_id"], score))

# sort descending
scores = sorted(scores, key=lambda x: x[1], reverse=True)

scores[:5]

[('0704.0027', np.float32(0.6025565)),
 ('0704.0025', np.float32(0.6005001)),
 ('0704.0021', np.float32(0.591717)),
 ('0704.0026', np.float32(0.5732733)),
 ('0704.0002', np.float32(0.57147247))]

In [23]:
top_k = 5

for pid, score in scores[:top_k]:
    row = batch[batch["id"] == pid].iloc[0]
    print(f"\nID: {pid}")
    print(f"Score: {score:.3f}")
    print(f"Title: {row['title']}")


ID: 0704.0027
Score: 0.603
Title: Filling-Factor-Dependent Magnetophonon Resonance in Graphene

ID: 0704.0025
Score: 0.601
Title: Spectroscopic Properties of Polarons in Strongly Correlated Systems by Exact Diagrammatic Monte Carlo Method

ID: 0704.0021
Score: 0.592
Title: Molecular Synchronization Waves in Arrays of Allosterically Regulated Enzymes

ID: 0704.0026
Score: 0.573
Title: Placeholder Substructures II: Meta-Fractals, Made of Box-Kites, Fill Infinite-Dimensional Skies

ID: 0704.0002
Score: 0.571
Title: Sparsity-certifying Graph Decompositions


## Segmented Retrieval (Prototype)

We implemented a multi-field retrieval system using separate embeddings for:

- title  
- abstract  
- comments  

Each field is embedded independently using Nomic embeddings.

At query time:
- the query is embedded once  
- similarity is computed with each field  
- scores are combined using a weighted sum  

score(q, d) =  
w_title * cos(q, title)  
+ w_abstract * cos(q, abstract)  
+ w_comments * cos(q, comments)

We tested this on a small batch of papers and successfully retrieved and ranked results.

Initial results show that the system captures general semantic signals (e.g. "graph", "molecular"), but may not yet fully capture more specific concepts (e.g. "neural networks"), highlighting the need for further evaluation and tuning.

In [24]:
# embed baseline texts (same batch)
baseline_embs = embed_texts(batch["baseline_text"].tolist(), prefix="search_document")

# compute scores
baseline_scores = []

for i in range(len(batch)):
    score = cosine_similarity(query_emb, baseline_embs[i])
    baseline_scores.append((batch.iloc[i]["id"], score))

# sort
baseline_scores = sorted(baseline_scores, key=lambda x: x[1], reverse=True)

# print top 5
top_k = 5
for pid, score in baseline_scores[:top_k]:
    row = batch[batch["id"] == pid].iloc[0]
    print(f"\nID: {pid}")
    print(f"Score: {score:.3f}")
    print(f"Title: {row['title']}")


ID: 0704.0021
Score: 0.642
Title: Molecular Synchronization Waves in Arrays of Allosterically Regulated Enzymes

ID: 0704.0025
Score: 0.618
Title: Spectroscopic Properties of Polarons in Strongly Correlated Systems by Exact Diagrammatic Monte Carlo Method

ID: 0704.0027
Score: 0.614
Title: Filling-Factor-Dependent Magnetophonon Resonance in Graphene

ID: 0704.0026
Score: 0.598
Title: Placeholder Substructures II: Meta-Fractals, Made of Box-Kites, Fill Infinite-Dimensional Skies

ID: 0704.0005
Score: 0.592
Title: From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$


### Preliminary Observation

On a small sample (32 papers), segmented retrieval produced results very similar to the baseline (title + abstract).

This suggests that:
- most semantic signal is already captured in the abstract
- simple field segmentation with fixed weights may not significantly improve retrieval

Further improvements may require:
- better field separation
- learned weighting
- larger-scale evaluation

So this is bullshit because its like the same as the baseline -> we need to get information from the pdfs themselves!

In [25]:
def arxiv_pdf_url(arxiv_id):
    return f"https://arxiv.org/pdf/{arxiv_id}.pdf"

batch["pdf_url"] = batch["id"].apply(arxiv_pdf_url)
batch[["id", "pdf_url"]].head(5)

/state/partition1/job-5777680/ipykernel_1347744/2580609829.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  batch["pdf_url"] = batch["id"].apply(arxiv_pdf_url)


,id,pdf_url
0,0704.0001,https://arxiv.org/pdf/0704.0001.pdf
1,0704.0002,https://arxiv.org/pdf/0704.0002.pdf
2,0704.0003,https://arxiv.org/pdf/0704.0003.pdf
3,0704.0004,https://arxiv.org/pdf/0704.0004.pdf
4,0704.0005,https://arxiv.org/pdf/0704.0005.pdf


In [26]:
import requests
from pathlib import Path
import os

PDF_DIR = Path(os.environ["SCRATCH"]) / "arxiv_pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

test_row = batch.iloc[0]
pdf_url = test_row["pdf_url"]
pdf_path = PDF_DIR / f"{test_row['id'].replace('/', '_')}.pdf"

resp = requests.get(pdf_url, timeout=60)
resp.raise_for_status()

with open(pdf_path, "wb") as f:
    f.write(resp.content)

print("Saved to:", pdf_path)
print("Size (MB):", round(pdf_path.stat().st_size / 1e6, 2))

Saved to: /scratch/mmd9604/arxiv_pdfs/0704.0001.pdf
Size (MB): 1.65


In [28]:
import fitz  # PyMuPDF

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = []
    for page in doc:
        text.append(page.get_text())
    return "\n".join(text)

raw_text = extract_pdf_text(pdf_path)

print(raw_text[:4000])

arXiv:0704.0001v2  [hep-ph]  24 Jul 2007
ANL-HEP-PR-07-12,
arXiv:0704.0001
Cal ulation
of
prompt
diphoton
pro
du tion
 ross
se tions
at
T
ev
atron
and
LHC
energies
C.
Balázs1
,∗
E.
L.
Berger1
,†
P
.
Nadolsky1
,‡
and
C.-P
.
Y
uan2§
1
High
Ener
gy
Physi s
Division,
A
r
gonne
National
L
ab
or
atory,
A
r
gonne,
IL
60439
2
Dep
artment
of
Physi s
and
Astr
onomy,
Mi higan
State
University,
East
L
ansing,
MI
48824
(Dated:
Ma
y
3,
2007)
Abstra t
A
fully
dieren
tial
 al ulation
in
p
erturbativ
e
quan
tum
 
hromo
dynami s
is
presen
ted
for
the
pro
du tion
of
massiv
e
photon
pairs
at
hadron
 olliders.
All
next-to-leading
order
p
erturbativ
e
 on
tributions
from
quark-an
tiquark,
gluon-(an
ti)quark,
and
gluon-gluon
subpro
 esses
are
in luded,
as
w
ell
as
all-orders
resummation
of
initial-state
gluon
radiation
v
alid
at
next-to-next-to-leading
logarithmi 
a  ura y
.
The
region
of
phase
spa e
is
sp
e ied
in
whi 
h
the
 al ulation
is
most
reliable.
Go
o
d
agreemen
t
is
demonstrated
with
data
from
th

In [29]:
import re

patterns = [
    r"\bCONCLUSION\b",
    r"\bCONCLUSIONS\b",
    r"\bDISCUSSION\b",
    r"\bSUMMARY\b",
    r"\bSUMMARY AND CONCLUSIONS\b",
]

for pat in patterns:
    matches = list(re.finditer(pat, raw_text, flags=re.IGNORECASE))
    print(f"{pat}: {len(matches)} match(es)")
    for m in matches[:3]:
        start = max(0, m.start() - 200)
        end = min(len(raw_text), m.end() + 500)
        print("\n---")
        print(raw_text[start:end])
        print("---\n")

\bCONCLUSION\b: 0 match(es)
\bCONCLUSIONS\b: 1 match(es)

---
n
the
presen
t
dis ussion,
w
e
w
ould
argue
that
the
resummed
QT
, ϕ∗
,
and cos θ∗
distributions
are
go
o
d
dis riminators
b
et
w
een
the
Higgs
b
oson
signal
and
ba 
kground
in
su 
h
an
analysis.
IV.
CONCLUSIONS
The
theoreti al
study
of
 on
tin
uum
diphoton
pro
du tion
in
hadron
 ollisions
is
in
teresting
and
v
aluable
for
sev
eral
reasons:
there
are
data
from
the
CDF
and
DØ
 ollab
orations
at
F
ermilab
with
the
promise
of
larger
ev
en
t
samples;
there
are
new
theoreti al
 
hallenges
asso
 iated
with
all-orders
soft-gluon
resummation
of
t
w
o-lo
op
amplitudes;
and
 on
tin
uum
diphotons
are
a
large
standard-mo
del
ba 
kground
ab
o
v
e
whi 
h
one
ma
y
observ
e
the
pro
du ts
of
Higgs
b
oson
de a
y
in
to
a
---

\bDISCUSSION\b: 0 match(es)
\bSUMMARY\b: 1 match(es)

---
 γγ
-fragmen
tation
ma
y
b
e
el-
ev
ated
in
some
kinemati 
regions
for
t
ypi al
exp
erimen
tal
 uts.
They
 an
b
e
remo
v
ed
b
y
adjustmen
ts
in
the
exp
erimen
tal


In [31]:
import re

def extract_section(text, start_pattern, end_patterns):
    start_match = re.search(start_pattern, text, flags=re.IGNORECASE)
    if not start_match:
        return None

    start_idx = start_match.end()
    remaining = text[start_idx:]

    end_idx = len(remaining)
    for pat in end_patterns:
        m = re.search(pat, remaining, flags=re.IGNORECASE)
        if m:
            end_idx = min(end_idx, m.start())

    return remaining[:end_idx].strip()

conclusion_text = extract_section(
    raw_text,
    start_pattern=r"(?:[IVXLC]+\.\s*)?CONCLUSIONS?",
    end_patterns=[
        r"(?:REFERENCES|ACKNOWLEDGMENTS?|APPENDIX)",
    ]
)

#print(conclusion_text[:3000] if conclusion_text else "No conclusion found")

In [34]:
paper_with_conclusion_0 = {
    "paper_id": test_row["id"],
    "title": batch.iloc[0]["title"],
    "abstract": batch.iloc[0]["abstract"],
    "comments": batch.iloc[0]["comments"] if pd.notna(batch.iloc[0]["comments"]) else "",
    "conclusion": conclusion_text,
}

paper_with_conclusion_0.keys()

dict_keys(['paper_id', 'title', 'abstract', 'comments', 'conclusion'])

In [35]:
conclusion_emb_0 = embed_texts([conclusion_text], prefix="search_document")[0]

paper_with_conclusion_emb_0 = {
    "paper_id": test_row["id"],
    "title": title_embs[0],
    "abstract": abstract_embs[0],
    "comments": comments_embs[0],
    "conclusion": conclusion_emb_0,
}

{k: v.shape if hasattr(v, "shape") else type(v) for k, v in paper_with_conclusion_emb_0.items()}

{'paper_id': str,
 'title': (768,),
 'abstract': (768,),
 'comments': (768,),
 'conclusion': (768,)}

In [36]:
# new weights (you can tweak later)
weights = {
    "title": 0.3,
    "abstract": 0.4,
    "comments": 0.1,
    "conclusion": 0.2,
}

# compute score for this one paper (just to test)
score_with_conclusion = (
    weights["title"] * cosine_similarity(query_emb, paper_with_conclusion_emb_0["title"]) +
    weights["abstract"] * cosine_similarity(query_emb, paper_with_conclusion_emb_0["abstract"]) +
    weights["comments"] * cosine_similarity(query_emb, paper_with_conclusion_emb_0["comments"]) +
    weights["conclusion"] * cosine_similarity(query_emb, paper_with_conclusion_emb_0["conclusion"])
)

score_with_conclusion

np.float32(0.47883236)

In [37]:
import re
import requests
from pathlib import Path
import fitz
import pandas as pd

PDF_DIR = Path(os.environ["SCRATCH"]) / "arxiv_pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = []
    for page in doc:
        text.append(page.get_text())
    return "\n".join(text)

def extract_section(text, start_pattern, end_patterns):
    start_match = re.search(start_pattern, text, flags=re.IGNORECASE)
    if not start_match:
        return ""
    start_idx = start_match.end()
    remaining = text[start_idx:]

    end_idx = len(remaining)
    for pat in end_patterns:
        m = re.search(pat, remaining, flags=re.IGNORECASE)
        if m:
            end_idx = min(end_idx, m.start())

    return remaining[:end_idx].strip()

def get_conclusion_text(arxiv_id):
    pdf_path = PDF_DIR / f"{arxiv_id.replace('/', '_')}.pdf"

    if not pdf_path.exists():
        url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        with open(pdf_path, "wb") as f:
            f.write(resp.content)

    raw_text = extract_pdf_text(pdf_path)

    conclusion = extract_section(
        raw_text,
        start_pattern=r"(?:[IVXLC]+\.\s*)?CONCLUSIONS?",
        end_patterns=[
            r"(?:REFERENCES|ACKNOWLEDGMENTS?|APPENDIX)",
        ]
    )

    return conclusion

# work on a small slice so this stays manageable
batch2 = batch.iloc[:10].copy()

# extract conclusions
batch2["conclusion"] = batch2["id"].apply(get_conclusion_text)

# embed all fields
title_embs2 = embed_texts(batch2["title"].tolist(), prefix="search_document")
abstract_embs2 = embed_texts(batch2["abstract"].tolist(), prefix="search_document")
comments_embs2 = embed_texts(batch2["comments"].fillna("").tolist(), prefix="search_document")
conclusion_embs2 = embed_texts(batch2["conclusion"].fillna("").tolist(), prefix="search_document")

# score and rank
weights = {
    "title": 0.3,
    "abstract": 0.4,
    "comments": 0.1,
    "conclusion": 0.2,
}

query = "graph neural networks for molecules"
query_emb = embed_texts([query], prefix="search_query")[0]

scores_with_conclusion = []
for i in range(len(batch2)):
    score = (
        weights["title"] * cosine_similarity(query_emb, title_embs2[i]) +
        weights["abstract"] * cosine_similarity(query_emb, abstract_embs2[i]) +
        weights["comments"] * cosine_similarity(query_emb, comments_embs2[i]) +
        weights["conclusion"] * cosine_similarity(query_emb, conclusion_embs2[i])
    )
    scores_with_conclusion.append((batch2.iloc[i]["id"], score))

scores_with_conclusion = sorted(scores_with_conclusion, key=lambda x: x[1], reverse=True)

for pid, score in scores_with_conclusion[:5]:
    row = batch2[batch2["id"] == pid].iloc[0]
    print(f"\nID: {pid}")
    print(f"Score: {score:.3f}")
    print(f"Title: {row['title']}")
    print(f"Conclusion chars: {len(row['conclusion'])}")

MuPDF error: format error: No default Layer config


ID: 0704.0002
Score: 0.575
Title: Sparsity-certifying Graph Decompositions
Conclusion chars: 2448

ID: 0704.0010
Score: 0.524
Title: Partial cubes: structures, characterizations, and constructions
Conclusion chars: 1179

ID: 0704.0006
Score: 0.522
Title: Bosonic characters of atomic Cooper pairs across resonance
Conclusion chars: 2761

ID: 0704.0007
Score: 0.493
Title: Polymer Quantum Mechanics and its Continuum Limit
Conclusion chars: 0

ID: 0704.0005
Score: 0.486
Title: From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$
Conclusion chars: 0
